# Final Decision Product

**Decision problem:** what recommendation can we make after running the full course pipeline?

This notebook consumes Bloc 1, Bloc 2, and Bloc 3 outputs and assembles the executive decision product.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]: d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"
def load_clean_long():
    p=OUT/"bloc1"/"clean_trends_long.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.melt(id_vars="date", var_name="signal", value_name="interest")
def load_clean_wide():
    p=OUT/"bloc1"/"clean_trends_wide.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.sort_values("date")

In [ ]:
def read_text(path, fallback="Missing. Run previous notebooks first."):
    return path.read_text() if path.exists() else fallback
summary = pd.read_csv(OUT/"bloc1"/"eda_signal_summary.csv", index_col=0) if (OUT/"bloc1"/"eda_signal_summary.csv").exists() else pd.DataFrame()
metrics = pd.read_csv(OUT/"bloc1"/"model_metrics.csv") if (OUT/"bloc1"/"model_metrics.csv").exists() else pd.DataFrame()
monitor = pd.read_csv(OUT/"bloc2"/"pipeline_monitoring_report.csv") if (OUT/"bloc2"/"pipeline_monitoring_report.csv").exists() else pd.DataFrame()
kpi = pd.read_csv(OUT/"bloc3"/"kpi_snapshot.csv") if (OUT/"bloc3"/"kpi_snapshot.csv").exists() else pd.DataFrame()
rec = read_text(OUT/"bloc3"/"final_recommendation.md")
print("Inputs loaded:", {"summary":not summary.empty,"metrics":not metrics.empty,"monitor":not monitor.empty,"kpi":not kpi.empty})

In [ ]:
executive_summary = f"""# Executive Summary

Recommendation: adopt a monthly weak-signal review as the first data product.

Evidence chain:
- Bloc 1 produced clean signals, EDA, feature tables, and model checks.
- Bloc 2 produced architecture, format, partition, and monitoring artifacts.
- Bloc 3 converted evidence into KPIs, BI outputs, experiment design, ROI, and recommendation.

Decision: start lightweight, monitor ChatGPT as the strategic weak signal, and validate action through one small experiment before investing in heavier forecasting infrastructure.

Limitations: Google Trends measures relative search interest, not demand or causality; model validation is small-sample; business impact requires experiment data.

Next steps: publish dashboard dataset, run monthly review, launch one A/B-tested content or discovery action.
"""
(OUT/"final_product"/"executive_summary.md").write_text(executive_summary)
print(executive_summary)

In [ ]:
if not summary.empty:
    summary.to_csv(OUT/"final_product"/"final_signal_summary.csv")
if not metrics.empty:
    metrics.to_csv(OUT/"final_product"/"final_model_metrics.csv", index=False)
if not monitor.empty:
    monitor.to_csv(OUT/"final_product"/"final_pipeline_status.csv", index=False)
if not kpi.empty:
    kpi.to_csv(OUT/"final_product"/"final_kpi_snapshot.csv", index=False)
manifest={"final_product_artifacts":[str(p.relative_to(ROOT)) for p in sorted((OUT/"final_product").glob("*"))]}
(OUT/"final_product"/"final_manifest.json").write_text(json.dumps(manifest, indent=2))
manifest

## Conclusion

The full course now behaves as one small data product pipeline: raw signal, trusted data, analysis, model checks, engineering artifacts, BI outputs, and a decision recommendation.